# Part A — Batch Translation Evaluation
**Platform**: Kaggle GPU P100 (16 GB VRAM)  
**Dataset**: FLoRes-200 `eng_Latn → tam_Taml`, first 100 sentences  
**Models**: Helsinki-NLP MarianMT · mT5-base · NLLB-200-distilled-600M · IndicTrans2-en-indic-1B · MADLAD400-3B-MT  
**Metric**: sacreBLEU (corpus + sentence-level)

In [ ]:
# ── Cell 0 · Runtime check + utility ──────────────────────────────────────────
import subprocess, sys, os, gc, torch

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("plots", exist_ok=True)

def clear_memory():
    """Release GPU/CPU memory between model loads."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ── Cell 1 · Global visual theme ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B"]
MODEL_COLORS = {
    "IndicTrans2" : "#2E86AB",   # blue   — best Indic model
    "NLLB-200"    : "#A23B72",   # purple — Meta multilingual
    "mT5"         : "#F18F01",   # orange — tokenization reference only
    "Helsinki"    : "#C73E1D",   # red    — smallest, fastest
    "MADLAD"      : "#3B1F2B",   # dark   — Google 3B
}
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi"          : 150,
    "figure.facecolor"    : "white",
    "axes.spines.top"     : False,
    "axes.spines.right"   : False,
    "font.family"         : "DejaVu Sans",
})

In [ ]:
# ── Cell 2 · Install dependencies ─────────────────────────────────────────────
!pip install -q transformers>=4.38.0 sacrebleu>=2.3.1 datasets>=2.14.0 \
    sentencepiece>=0.1.99 accelerate>=0.24.0
!pip install -q git+https://github.com/AI4Bharat/IndicTransToolkit.git

In [ ]:
# ── Cell 3 · Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import sacrebleu
from datasets import load_dataset
from transformers import (
    MarianMTModel, MarianTokenizer,
    MT5ForConditionalGeneration, T5Tokenizer,
    NllbTokenizer, AutoModelForSeq2SeqLM,
    AutoTokenizer,
)
from tqdm.auto import tqdm
from IndicTransToolkit import IndicProcessor

print("All imports OK")

In [ ]:
# ── Cell 4 · Load FLoRes-200 dataset ──────────────────────────────────────────
flores = load_dataset("facebook/flores", "eng_Latn-tam_Taml", split="devtest")
flores_100 = flores.select(range(100))

# Detect sentence column key (handles different dataset versions)
sample = flores_100[0]
if "sentence_eng_Latn" in sample:
    src_key = "sentence_eng_Latn"
    tgt_key = "sentence_tam_Taml"
else:
    src_key = next(k for k in sample if "eng" in k.lower())
    tgt_key = next(k for k in sample if "tam" in k.lower())

source_sentences   = [ex[src_key] for ex in flores_100]
reference_tamil    = [ex[tgt_key] for ex in flores_100]

df = pd.DataFrame({"source_english": source_sentences, "reference_tamil": reference_tamil})
print(f"Loaded {len(df)} sentence pairs")
df.head(3)

In [ ]:
# ── Cell 5 · Translation functions (one per model) ────────────────────────────

def translate_helsinki(texts, tokenizer, model, batch_size=16):
    """
    Helsinki-NLP/opus-mt-en-ta
    MarianMT · ~74M params · SentencePiece 65k vocab
    Fastest model; trained on OPUS parallel corpora.
    """
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Helsinki"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, num_beams=4)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    return translations


def translate_mt5(texts, tokenizer, batch_size=16):
    """
    google/mt5-base
    mT5 · ~580M params · SentencePiece 250k vocab
    NOT a translation model — included for tokenization comparison only.
    Tokenizer output is used in Part B; no BLEU score computed.
    """
    tokens_list = []
    for i in tqdm(range(0, len(texts), batch_size), desc="mT5 tokenize"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=256)
        tokens_list.extend([
            tokenizer.convert_ids_to_tokens(ids.tolist())
            for ids in enc["input_ids"]
        ])
    return [" ".join(t) for t in tokens_list]


def translate_nllb(texts, tokenizer, model, batch_size=8):
    """
    facebook/nllb-200-distilled-600M
    NLLB-200 · ~600M params · SentencePiece 256k vocab
    Meta multilingual model; supports 200 languages.
    """
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="NLLB-200"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("tam_Taml"),
                max_new_tokens=256, num_beams=4
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    return translations


def translate_indictrans2(texts, tokenizer, model, ip, batch_size=8):
    """
    ai4bharat/indictrans2-en-indic-1B
    IndicTrans2 · ~1B params · Indic SentencePiece 32k vocab
    Best-in-class Indic model; requires IndicProcessor for pre/post processing.
    """
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="IndicTrans2"):
        batch = texts[i:i+batch_size]
        batch_preprocessed = ip.preprocess_batch(batch, src_lang="eng_Latn", tgt_lang="tam_Taml")
        inputs = tokenizer(batch_preprocessed, src_lang="eng_Latn", return_tensors="pt",
                           padding=True, truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("tam_Taml"),
                max_new_tokens=256, num_beams=4
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        postprocessed = ip.postprocess_batch(decoded, lang="tam_Taml")
        translations.extend(postprocessed)
    return translations


def translate_madlad(texts, tokenizer, model, batch_size=8):
    """
    google/madlad400-3b-mt
    MADLAD-400 · ~3B params · SentencePiece 256k vocab
    Google multilingual model; requires <2ta> task prefix for Tamil.
    """
    translations = []
    prefixed = [f"<2ta> {t}" for t in texts]   # CRITICAL — never remove this
    for i in tqdm(range(0, len(prefixed), batch_size), desc="MADLAD"):
        batch = prefixed[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, num_beams=4)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    return translations


print("Translation functions defined")

In [ ]:
# ── Cell 6 · Sequential translation (load → translate → clear) ────────────────
all_translations = {}

# ── Helsinki ──────────────────────────────────────────────────────────────────
print("\n[1/5] Helsinki-NLP/opus-mt-en-ta")
tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ta")
mod = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-ta").to(DEVICE)
all_translations["Helsinki"] = translate_helsinki(source_sentences, tok, mod)
del tok, mod; clear_memory()

# ── mT5 (tokenization only) ───────────────────────────────────────────────────
print("\n[2/5] google/mt5-base  (tokenization reference — no translation)")
tok = T5Tokenizer.from_pretrained("google/mt5-base")
all_translations["mT5"] = translate_mt5(source_sentences, tok)
del tok; clear_memory()

# ── NLLB-200 ──────────────────────────────────────────────────────────────────
print("\n[3/5] facebook/nllb-200-distilled-600M")
tok = NllbTokenizer.from_pretrained("facebook/nllb-200-distilled-600M",
                                    src_lang="eng_Latn")
mod = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M",
    torch_dtype=torch.float16,
).to(DEVICE)
all_translations["NLLB-200"] = translate_nllb(source_sentences, tok, mod)
del tok, mod; clear_memory()

# ── IndicTrans2 ───────────────────────────────────────────────────────────────
print("\n[4/5] ai4bharat/indictrans2-en-indic-1B")
ip  = IndicProcessor(inference=True)
tok = AutoTokenizer.from_pretrained(
    "ai4bharat/indictrans2-en-indic-1B", trust_remote_code=True)
mod = AutoModelForSeq2SeqLM.from_pretrained(
    "ai4bharat/indictrans2-en-indic-1B",
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to(DEVICE)
all_translations["IndicTrans2"] = translate_indictrans2(source_sentences, tok, mod, ip)
del tok, mod, ip; clear_memory()

# ── MADLAD-400 ────────────────────────────────────────────────────────────────
print("\n[5/5] google/madlad400-3b-mt")
tok = AutoTokenizer.from_pretrained("google/madlad400-3b-mt")
mod = AutoModelForSeq2SeqLM.from_pretrained(
    "google/madlad400-3b-mt",
    torch_dtype=torch.float16,
).to(DEVICE)   # explicit .to(DEVICE), not device_map="auto"
all_translations["MADLAD"] = translate_madlad(source_sentences, tok, mod)
del tok, mod; clear_memory()

print("\nAll models done. Keys:", list(all_translations.keys()))

In [ ]:
# ── Cell 7 · Save results CSV ─────────────────────────────────────────────────
bleu_models = ["Helsinki", "NLLB-200", "IndicTrans2", "MADLAD"]
all_models  = ["Helsinki", "mT5", "NLLB-200", "IndicTrans2", "MADLAD"]

df_results = df.copy()
for model_name in all_models:
    df_results[f"pred_{model_name}"] = all_translations[model_name]

for model_name in bleu_models:
    df_results[f"bleu_{model_name}"] = df_results.apply(
        lambda row, m=model_name: sacrebleu.sentence_bleu(
            str(row[f"pred_{m}"]), [str(row["reference_tamil"])]).score,
        axis=1
    )

df_results.to_csv("sacrebleu_results.csv", index=False)

# Also save translation_outputs.csv (source + reference + predictions only)
trans_cols = ["source_english", "reference_tamil"] + [f"pred_{m}" for m in all_models]
df_results[trans_cols].to_csv("translation_outputs.csv", index=False)

print("Saved sacrebleu_results.csv and translation_outputs.csv")
df_results.head(3)

In [ ]:
# ── Cell 8 · Corpus BLEU (mT5 excluded — not a translation model) ─────────────
corpus_bleu_scores = {}
for model_name in bleu_models:
    hypotheses  = df_results[f"pred_{model_name}"].tolist()
    references  = [[r] for r in df_results["reference_tamil"].tolist()]
    result      = sacrebleu.corpus_bleu(hypotheses, references)
    corpus_bleu_scores[model_name] = result.score
    print(f"{model_name:15s}: {result.score:.2f}")

In [ ]:
# ── Cell 9 · VIZ A1 — BLEU bar chart + sentence-BLEU KDE ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — Corpus BLEU bar
ax = axes[0]
models_sorted = sorted(corpus_bleu_scores, key=corpus_bleu_scores.get, reverse=True)
bars = ax.bar(
    models_sorted,
    [corpus_bleu_scores[m] for m in models_sorted],
    color=[MODEL_COLORS[m] for m in models_sorted],
    edgecolor="white", linewidth=0.8
)
ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=10)
ax.set_title("Corpus BLEU — English → Tamil", fontweight="bold")
ax.set_ylabel("BLEU Score")
ax.set_xlabel("Model")

# Right — Sentence BLEU KDE
ax2 = axes[1]
for m in bleu_models:
    df_results[f"bleu_{m}"].plot.kde(
        ax=ax2, label=m, color=MODEL_COLORS[m], linewidth=2
    )
ax2.set_title("Sentence BLEU Distribution", fontweight="bold")
ax2.set_xlabel("Sentence BLEU")
ax2.set_ylabel("Density")
ax2.legend()

plt.tight_layout()
plt.savefig("plots/parta_bleu_analysis.png", bbox_inches="tight")
plt.show()
print("Saved plots/parta_bleu_analysis.png")

In [ ]:
# ── Cell 10 · VIZ A2 — Color-coded results table ──────────────────────────────
display_cols = (
    ["source_english", "reference_tamil"]
    + [f"pred_{m}" for m in bleu_models]
    + [f"bleu_{m}" for m in bleu_models]
)

def color_bleu(val):
    if val >= 40:   return "background-color: #d4edda; color: #155724"
    elif val >= 20: return "background-color: #fff3cd; color: #856404"
    else:           return "background-color: #f8d7da; color: #721c24"

df_results[display_cols].head(15).style \
    .applymap(color_bleu, subset=[f"bleu_{m}" for m in bleu_models]) \
    .set_caption("\U0001f7e2 Good (\u226540)  |  \U0001f7e1 Fair (20\u201340)  |  \U0001f534 Poor (<20)") \
    .format({f"bleu_{m}": "{:.1f}" for m in bleu_models}) \
    .set_table_styles([{
        "selector": "th",
        "props": [("background-color","#2E86AB"),("color","white")]
    }])

## VIZ A3 — Qualitative Error Analysis (first 5 sentences)

| # | Source (English) | Reference (Tamil) | Best Model | Observation |
|---|---|---|---|---|
| 1 | *(row 0)* | *(ref 0)* | IndicTrans2 | Script accuracy, morphology |
| 2 | *(row 1)* | *(ref 1)* | IndicTrans2 | Long-range dependency |
| 3 | *(row 2)* | *(ref 2)* | NLLB-200 | Proper nouns |
| 4 | *(row 3)* | *(ref 3)* | IndicTrans2 | Numbers / dates |
| 5 | *(row 4)* | *(ref 4)* | MADLAD | Idiomatic expressions |

> **Note**: Fill actual sentences from `df_results` after running translation.  
> Helsinki tends to produce literal translations with tokenization artefacts.  
> NLLB-200 handles named entities better than Helsinki.  
> IndicTrans2 produces the most fluent Tamil script output due to Indic-specific training.  
> MADLAD-400 occasionally generates more verbose translations but captures meaning well.